#### Understanding the Expert Cache and Scatter Reduce in MoE Inference
- Expert_cache which is of the same shape as x_flat is. The aggregate of the expert outputs per token. But in the moe_infer method, we are calculating the outputs of all tokens per expert. 
- Now we want to put the output of the corresponding token in its respective place in teh expert cache, so that when the for loop ends we have accumulated the outputs per token in the expert cache for all experts active on that particular token. For this purpose we are using the scatter_reduce functionality.
- For every iteration, the expert output contains the values for all tokens for which that particular expert is active.
The index specifies where in the expert cache are we supposed to deposit token wise values.
- Now lets say for a particular expert 14 tokens were getting passed, so the expert_out will have the shape of [14,128]. and we know that the exp_token_idx has the index of all the tokens aactive for this expert which is a 1d tensor.
But now that we have to put each token (with 128 values) in its correct place, we transform teh exp_token_idxs to shape [14,128] from [14]
- So now we know that for every token from the expert_out, where do i have to deposit its 128 values, we have one index for each of the 128 values in the index tensor.

*expert_cache*:
- Initialized with the same shape as x_flat, i.e. (batch * seq, d_hidden).
- It’s the accumulator where each token’s expert contributions are deposited.

*Loop in moe_infer*:
- Instead of looping token‑by‑token, it loops expert‑by‑expert.- For each expert, you gather all tokens routed to it (exp_token_idx).
- You run those tokens through the expert → expert_out of shape (n_tokens_for_expert, d_hidden).

*Scatter‑reduce*:
- You need to put each row of expert_out back into the correct slot in expert_cache.
- exp_token_idx is (n_tokens_for_expert,).
- To align with expert_out (n_tokens_for_expert, d_hidden), you broadcast/expand exp_token_idx to (n_tokens_for_expert, d_hidden).
- Now each of the 128 hidden‑dim values for a token knows exactly which position in expert_cache to go to.
- scatter_reduce_ then deposits them, summing if multiple experts contribute to the same token.

In [1]:
import torch
import numpy as np
import torch.nn as nn
from typing import Dict, Tuple, List, Optional
import math

In [45]:
config = {"dhidden": 16, "dexpert": 4, "nroutedexperts": 4,
          "nsharedexperts": 1, "nexpertspertoken": 2, "bs": 2, "seqlen": 4, "seed": 9371}

gen = torch.Generator(device='cuda')
gen.manual_seed(config["seed"])

x = torch.randn(
        (config["bs"], config["seqlen"], config["dhidden"]),
        device='cuda',
        dtype=torch.float16,
        generator=gen
    ).contiguous()

print(x.shape)

torch.Size([2, 4, 16])


In [7]:
config = {"dhidden": 7168, "dexpert": 2048, "nroutedexperts": 8,
          "nsharedexperts": 1, "nexpertspertoken": 4, "bs": 2, "seqlen": 512, "seed": 9371}

gen = torch.Generator(device='cuda')
gen.manual_seed(config["seed"])

x = torch.randn(
        (config["bs"], config["seqlen"], config["dhidden"]),
        device='cuda',
        dtype=torch.float16,
        generator=gen
    ).contiguous()

print(x.shape)

torch.Size([2, 512, 7168])


In [46]:
x.view(-1, config["dhidden"])[0].shape

torch.Size([16])

In [47]:
token = x[1,2]
print(token.shape)
token = x[1,2].unsqueeze(0)
print(token.shape)

torch.Size([16])
torch.Size([1, 16])


In [48]:
router_weights = torch.randn(
        (config["nroutedexperts"], config["dhidden"]),
        device="cuda",
        dtype=torch.float16,
        generator=gen
    ) / math.sqrt( config["dhidden"])
router_weights.shape

torch.Size([4, 16])

In [49]:
def _stack_t(weights, key_tmpl: str, n, out_shape):
    mats = [weights[key_tmpl.format(i)].t() for i in range(n)]
    return torch.stack(mats, dim=0).reshape(out_shape).contiguous()

weights = {}
for i in range(config["nroutedexperts"]):
        weights[f'experts.{i}.0.weight'] = torch.randn(
            (config["dhidden"], config["dexpert"]),
            device='cuda',
            dtype=torch.float16,
            generator=gen
        ) / math.sqrt(config["dexpert"])
print(weights['experts.0.0.weight'].shape)
W_gate = nn.Parameter(
            torch.empty(config["nroutedexperts"], config["dexpert"], config["dhidden"], device='cuda', dtype=torch.float16)
        )
W_gate.data.copy_(
            _stack_t(weights, "experts.{}.0.weight", config["nroutedexperts"], (config["nroutedexperts"], config["dexpert"], config["dhidden"])).to(
                "cuda", torch.float16
            )
        )
print(W_gate.shape)

# mats = [weights['experts.{}.0.weight'.format(i)].t() for i in range(config["nroutedexperts"])]
# print(mats[0].shape)
# torch.stack(mats, dim=0).shape#reshape((config["nroutedexperts"], config["dexpert"], config["dhidden"])).shape

torch.Size([16, 4])
torch.Size([4, 4, 16])


In [97]:
W_g = nn.Linear(config["dhidden"], config["nroutedexperts"], bias=False)
W_g.weight = nn.Parameter(router_weights)
logits = W_g(x)
scores = logits.softmax(dim= -1)
topk_scores, topk_indi = torch.topk(scores, k = config["nexpertspertoken"], sorted= False)

x_flat = x.view(-1, config["dhidden"])
flat_exp_indi = topk_indi.view(-1)
flat_exp_weights = topk_scores.view(-1, 1)

print(f"x shape {x.shape}")
print(f"x_flat shape {x_flat.shape}")
print(f"scores shape {scores.shape}")
print(f"topk_indi shape {topk_indi.shape}")
print(f"topk_scores shape {topk_scores.shape}")
print(f"flat_exp_indi shape {flat_exp_indi.shape}")
print(f"flat_exp_weights shape {flat_exp_weights.shape}")

x shape torch.Size([2, 4, 16])
x_flat shape torch.Size([8, 16])
scores shape torch.Size([2, 4, 4])
topk_indi shape torch.Size([2, 4, 2])
topk_scores shape torch.Size([2, 4, 2])
flat_exp_indi shape torch.Size([16])
flat_exp_weights shape torch.Size([16, 1])


In [73]:
print(scores)
print(topk_scores)
print(topk_indi)
print(flat_exp_weights)

tensor([[[0.8750, 0.0180, 0.0839, 0.0232],
         [0.4546, 0.1825, 0.1462, 0.2166],
         [0.3286, 0.0614, 0.3264, 0.2837],
         [0.3491, 0.1017, 0.2170, 0.3320]],

        [[0.2595, 0.3091, 0.1174, 0.3140],
         [0.7012, 0.0735, 0.1202, 0.1050],
         [0.1161, 0.2328, 0.6284, 0.0225],
         [0.2113, 0.0426, 0.0401, 0.7061]]], device='cuda:0',
       dtype=torch.float16, grad_fn=<SoftmaxBackward0>)
tensor([[[0.8750, 0.0839],
         [0.4546, 0.2166],
         [0.3286, 0.3264],
         [0.3491, 0.3320]],

        [[0.3140, 0.3091],
         [0.7012, 0.1202],
         [0.6284, 0.2328],
         [0.7061, 0.2113]]], device='cuda:0', dtype=torch.float16,
       grad_fn=<TopkBackward0>)
tensor([[[0, 2],
         [0, 3],
         [0, 2],
         [0, 3]],

        [[3, 1],
         [0, 2],
         [2, 1],
         [3, 0]]], device='cuda:0')
tensor([[0.8750],
        [0.0839],
        [0.4546],
        [0.2166],
        [0.3286],
        [0.3264],
        [0.3491],
      

In [94]:
print(f"flat_exp_indicies : {flat_exp_indi}")
counts = flat_exp_indi.bincount().cpu().numpy()
print(f"counts: {counts}")
tokens_per_exp = counts.cumsum()
print(f"tokens per expert: {tokens_per_exp}")


flat_exp_indicies : tensor([0, 2, 0, 3, 0, 2, 0, 3, 3, 1, 0, 2, 2, 1, 3, 0], device='cuda:0')
counts: [6 2 4 4]
tokens per expert: [ 6  8 12 16]


In [78]:
print(flat_exp_indi)
idxs = flat_exp_indi.argsort()
print(idxs)
token_idxs = idxs // config["nexpertspertoken"]
print(token_idxs)

tensor([0, 2, 0, 3, 0, 2, 0, 3, 3, 1, 0, 2, 2, 1, 3, 0], device='cuda:0')
tensor([ 6,  4,  0,  2, 10, 15, 13,  9,  5,  1, 11, 12, 14,  8,  3,  7],
       device='cuda:0')
tensor([3, 2, 0, 1, 5, 7, 6, 4, 2, 0, 5, 6, 7, 4, 1, 3], device='cuda:0')


In [96]:
sorted_exp_ids = flat_exp_indi[idxs]
sorted_exp_ids

tensor([0, 0, 0, 0, 0, 0, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3], device='cuda:0')

In [90]:
config = {"dhidden": 16, "dexpert": 4, "nroutedexperts": 4,
          "nsharedexperts": 1, "nexpertspertoken": 2, "bs": 2, "seqlen": 4, "seed": 9371}

num_tokens = config["bs"] * config["seqlen"]
token_ind_flat_pair = torch.arange(num_tokens, device="cuda", dtype=torch.long).unsqueeze(1).expand(-1, config['nexpertspertoken']).reshape(-1)
token_ind_flat_pair

tensor([0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7], device='cuda:0')

In [104]:
weights = {}

def _stack_t(self, weights, key_tmpl: str, n, out_shape):
        mats = [weights[key_tmpl.format(i)].t() for i in range(n)]
        return torch.stack(mats, dim=0).reshape(out_shape).contiguous()

for i in range(config["nroutedexperts"]):
        weights[f'experts.{i}.0.weight'] = torch.randn(
            (config["dhidden"], config["dexpert"]),
            device='cuda',
            dtype=torch.float16,
            generator=gen
        ) / math.sqrt(config["dexpert"])

print(weights["experts.0.0.weight"].shape)

mats = [weights[f"experts.{i}.0.weight".format(i)].t() for i in range(config["nroutedexperts"])]
print(mats[0].shape)

torch.stack(mats, dim=0).reshape(config["nroutedexperts"], config["dexpert"], config["dhidden"]).contiguous().shape


torch.Size([16, 4])
torch.Size([4, 16])


torch.Size([4, 4, 16])

In [95]:
perm =torch.argsort(flat_exp_indi, stable=True, dim=-1).int()
print(perm)
print(flat_exp_indi[perm])
gather_indi = token_ind_flat_pair[perm]
gather_indi

tensor([ 0,  2,  4,  6, 10, 15,  9, 13,  1,  5, 11, 12,  3,  7,  8, 14],
       device='cuda:0', dtype=torch.int32)
tensor([0, 0, 0, 0, 0, 0, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3], device='cuda:0')


tensor([0, 1, 2, 3, 5, 7, 4, 6, 0, 2, 5, 6, 1, 3, 4, 7], device='cuda:0')

In [62]:
class Expert(nn.Module):
    def __init__(self, config: Dict, dexpert: Optional[int] = None):
        super().__init__()
        self.act_fn = nn.SiLU()
        self.dhidden: int = config["dhidden"]
        self.dexpert: int = config["dexpert" if dexpert is None else dexpert]

        self.W_gate = nn.Linear(self.dhidden, self.dexpert, bias=False)
        self.W_up = nn.Linear(self.dhidden, self.dexpert, bias=False)
        self.W_down = nn.Linear(self.dexpert, self.dhidden, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = self.act_fn(self.W_gate(x))
        out = self.W_down(gate * self.W_up(x))
        return out


In [65]:
ex = Expert(config=config)
ex.W_down.weight.shape

torch.Size([128, 64])

In [25]:
# x_flat = x.view(-1, 128)
experts = nn.ModuleList([
    Expert(config)
    for _ in range(config["num_experts"])
])
expert_cache = torch.zeros_like(x_flat)  # shape (2,8,128)
for exp_id, end_idx in enumerate(tokens_per_exp):
    start_idx = 0 if exp_id == 0 else tokens_per_exp[exp_id - 1]
    if start_idx == end_idx: continue
    print("------------------")
    expert = experts[exp_id]
    print(f"Expert_ID: {exp_id}, Start_Index: {start_idx}, End_Index: {end_idx}")
    exp_token_idxs = token_idxs[start_idx:end_idx]
    expert_tokens = x_flat[exp_token_idxs]
    print(f"Expert_tokens shape: {expert_tokens.shape}")
    expert_out = expert(expert_tokens)
    expert_out.mul_(flat_exp_weights[idxs[start_idx:end_idx]])
    print(f"Expert Out shape: {expert_out.shape}")
    expert_cache.scatter_reduce_(
        0,
        exp_token_idxs.view(-1, 1).repeat(1, x.shape[-1]),
        expert_out,
        reduce='sum'
    )

------------------
Expert_ID: 0, Start_Index: 0, End_Index: 6
Expert_tokens shape: torch.Size([6, 128])
Expert Out shape: torch.Size([6, 128])
------------------
Expert_ID: 2, Start_Index: 6, End_Index: 7
Expert_tokens shape: torch.Size([1, 128])
Expert Out shape: torch.Size([1, 128])
------------------
Expert_ID: 4, Start_Index: 7, End_Index: 8
Expert_tokens shape: torch.Size([1, 128])
Expert Out shape: torch.Size([1, 128])
------------------
Expert_ID: 5, Start_Index: 8, End_Index: 13
Expert_tokens shape: torch.Size([5, 128])
Expert Out shape: torch.Size([5, 128])
------------------
Expert_ID: 7, Start_Index: 13, End_Index: 17
Expert_tokens shape: torch.Size([4, 128])
Expert Out shape: torch.Size([4, 128])
------------------
Expert_ID: 8, Start_Index: 17, End_Index: 25
Expert_tokens shape: torch.Size([8, 128])
Expert Out shape: torch.Size([8, 128])
------------------
Expert_ID: 10, Start_Index: 25, End_Index: 29
Expert_tokens shape: torch.Size([4, 128])
Expert Out shape: torch.Size([

In [26]:
expert_cache.scatter_reduce_(
        0,
        exp_token_idxs.view(-1, 1).repeat(1, x.shape[-1]),
        expert_out,
        reduce='sum'
    )
expert_cache.shape

torch.Size([16, 128])

In [33]:
# Expert_ID : 0
exp_token_idxs = token_idxs[0:6]
expert_tokens = x_flat[exp_token_idxs]
print(f"Expert_tokens shape: {expert_tokens.shape}")
expert_out = expert(expert_tokens)
expert_out.mul_(flat_exp_weights[idxs[0:6]])
print(f"Expert Out shape: {expert_out.shape}")

Expert_tokens shape: torch.Size([6, 128])
Expert Out shape: torch.Size([6, 128])


In [42]:
expert_out.shape

torch.Size([6, 128])

In [36]:
exp_token_idxs

tensor([15, 11, 14,  5,  2, 13])

In [49]:
exp_token_idxs.view(-1,1).repeat(1, x_flat.shape[-1])

tensor([[15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15,
         15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15,
         15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15,
         15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15,
         15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15,
         15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15,
         15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15,
         15, 15],
        [11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         1